# InternVL3 Hybrid Processor Test

**SAFE TESTING**: Tests the new hybrid processor (InternVL3 model + Llama pipeline) without affecting existing code.

**Architecture**: 
- InternVL3 model for accuracy
- Llama's proven processing pipeline for reliability
- ExtractionCleaner for 🧹 CLEANER CALLED output
- Working 82% prompts (reverted from simple format)

**Expected Results**: Should combine the best of both worlds - InternVL3 accuracy + Llama reliability.

In [3]:
# Setup and imports with proper environment detection
import sys
import os
from pathlib import Path
import time

# CRITICAL FIX: Detect environment and set correct paths
current_dir = os.getcwd()
if '/home/jovyan/nfs_share/tod' in current_dir:
    # AISandbox environment
    project_root = '/home/jovyan/nfs_share/tod/LMM_POC'
    deployment_env = "AISandbox"
    models_base = '/home/jovyan/nfs_share/models'
elif '/efs/shared' in current_dir:
    # EFS environment
    project_root = '/efs/shared/LMM_POC'
    deployment_env = "efs"
    models_base = '/efs/shared/PTM'
else:
    # Local development - use current directory
    project_root = str(Path.cwd())
    deployment_env = "local"
    models_base = f"{project_root}/models"

# Add project root to path
sys.path.insert(0, project_root)
os.chdir(project_root)

print(f"📍 Project root: {project_root}")
print(f"🌍 Detected environment: {deployment_env}")
print(f"🚀 Models base: {models_base}")
print(f"🐍 Python path: {sys.path[0]}")
print(f"📂 Working directory: {os.getcwd()}")

# CRITICAL: Override deployment configuration
os.environ['CURRENT_DEPLOYMENT'] = deployment_env
print(f"✅ Set CURRENT_DEPLOYMENT={deployment_env}")

📍 Project root: /home/jovyan/nfs_share/tod/LMM_POC
🌍 Detected environment: AISandbox
🚀 Models base: /home/jovyan/nfs_share/models
🐍 Python path: /home/jovyan/nfs_share/tod/LMM_POC
📂 Working directory: /home/jovyan/nfs_share/tod/LMM_POC
✅ Set CURRENT_DEPLOYMENT=AISandbox


In [4]:
# Import the hybrid processor
try:
    from models.document_aware_internvl3_hybrid_processor import DocumentAwareInternVL3HybridProcessor
    print("✅ InternVL3 Hybrid Processor imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("💡 Check that the project root path is correct")
    raise

✅ InternVL3 Hybrid Processor imported successfully


In [5]:
# Configuration with model path validation
# Start with small field list for initial testing
TEST_FIELDS_SMALL = [
    "DOCUMENT_TYPE",
    "SUPPLIER_NAME", 
    "TOTAL_AMOUNT"
]

TEST_FIELDS_MEDIUM = [
    "DOCUMENT_TYPE",
    "SUPPLIER_NAME",
    "BUSINESS_ADDRESS",
    "TOTAL_AMOUNT",
    "GST_AMOUNT",
    "INVOICE_DATE"
]

# CRITICAL FIX: Validate model path before proceeding
from common.config import INTERNVL3_MODEL_PATH, configure_for_deployment

# Configure for detected deployment environment
configure_for_deployment(deployment_env)

print(f"🎯 Small test fields: {TEST_FIELDS_SMALL}")
print(f"🎯 Medium test fields: {TEST_FIELDS_MEDIUM}")
print(f"🚀 InternVL3 model path: {INTERNVL3_MODEL_PATH}")

# Validate model path exists
model_path = Path(INTERNVL3_MODEL_PATH)
if model_path.exists():
    print("✅ InternVL3 model path validated")
    # Look for key model files
    config_file = model_path / "config.json"
    if config_file.exists():
        print("✅ Model config.json found")
    else:
        print("⚠️ Model config.json not found - may need to check model installation")
else:
    print(f"❌ CRITICAL: Model path does not exist: {INTERNVL3_MODEL_PATH}")
    print("💡 Available paths to check:")
    for potential_path in ['/home/jovyan/nfs_share/models', '/efs/shared/PTM', 'models']:
        if Path(potential_path).exists():
            print(f"   - {potential_path} ✅")
            if (Path(potential_path) / "InternVL3-8B").exists():
                print(f"     └─ InternVL3-8B ✅")
            if (Path(potential_path) / "InternVL3-2B").exists():
                print(f"     └─ InternVL3-2B ✅")
        else:
            print(f"   - {potential_path} ❌")

# Test image paths - adjust based on environment
if deployment_env == "AISandbox":
    TEST_IMAGE_PATH = "/home/jovyan/nfs_share/tod/evaluation_data/image_001.png"
elif deployment_env == "efs":
    TEST_IMAGE_PATH = "/efs/shared/evaluation_data/image_001.png"
else:
    TEST_IMAGE_PATH = "evaluation_data/sample_invoice.jpg"

print(f"🖼️ Test image: {TEST_IMAGE_PATH}")
if Path(TEST_IMAGE_PATH).exists():
    print("✅ Test image found")
else:
    print("⚠️ Test image not found - will search for alternatives")

ImportError: cannot import name 'configure_for_deployment' from 'common.config' (/home/jovyan/nfs_share/tod/LMM_POC/common/config.py)

## Phase 1: Basic Initialization Test

In [ ]:
# Test basic initialization (small field list) with enhanced error handling
print("🚀 Testing hybrid processor initialization...")

try:
    # CRITICAL FIX: Validate deployment configuration first
    from common.config import INTERNVL3_MODEL_PATH, CURRENT_DEPLOYMENT
    print(f"🌍 Configuration: deployment={CURRENT_DEPLOYMENT}, model_path={INTERNVL3_MODEL_PATH}")
    
    # Validate model path exists before attempting to load
    model_path_obj = Path(INTERNVL3_MODEL_PATH)
    if not model_path_obj.exists():
        print(f"❌ CRITICAL: Model path does not exist: {INTERNVL3_MODEL_PATH}")
        print("🔧 Attempting to find model in alternative locations...")
        
        # Check alternative paths
        alternative_paths = [
            '/home/jovyan/nfs_share/models/InternVL3-8B',
            '/home/jovyan/nfs_share/models/InternVL3-2B',
            '/efs/shared/PTM/InternVL3-8B',
            '/efs/shared/PTM/InternVL3-2B'
        ]
        
        found_model = None
        for alt_path in alternative_paths:
            if Path(alt_path).exists():
                print(f"✅ Found model at: {alt_path}")
                found_model = alt_path
                break
        
        if found_model:
            print(f"🔧 Using alternative model path: {found_model}")
            # Override the model path
            import os
            os.environ['INTERNVL3_MODEL_PATH'] = found_model
            # Reload config
            from importlib import reload
            import common.config
            reload(common.config)
            from common.config import INTERNVL3_MODEL_PATH
            print(f"✅ Updated model path: {INTERNVL3_MODEL_PATH}")
        else:
            print("❌ No InternVL3 model found in any expected location")
            print("💡 Please ensure InternVL3 model is downloaded and available")
            raise FileNotFoundError(f"InternVL3 model not found at {INTERNVL3_MODEL_PATH}")
    
    # Initialize hybrid processor with error handling
    hybrid_processor = DocumentAwareInternVL3HybridProcessor(
        field_list=TEST_FIELDS_SMALL,
        debug=True  # Enable verbose output including 🧹 CLEANER CALLED
    )
    
    print("\n✅ HYBRID PROCESSOR INITIALIZED SUCCESSFULLY")
    print(f"📊 Field count: {hybrid_processor.field_count}")
    print(f"🎯 Model path: {hybrid_processor.model_path}")
    print(f"🧹 ExtractionCleaner: {'✅ Active' if hybrid_processor.cleaner else '❌ Missing'}")
    print(f"🔧 Device: {hybrid_processor.device}")
    print(f"🚀 Model loaded: {'✅ Yes' if hybrid_processor.model else '❌ No'}")
    
    # Additional model info
    model_info = hybrid_processor.get_model_info()
    print(f"📋 Model info: {model_info}")
    
except Exception as e:
    print(f"❌ Initialization failed: {e}")
    import traceback
    traceback.print_exc()
    
    # Provide helpful debugging information
    print("\n🔍 DEBUGGING INFO:")
    print(f"   Current working directory: {os.getcwd()}")
    print(f"   Python path: {sys.path[0]}")
    
    # Check for required imports
    try:
        import torch
        print(f"   PyTorch: ✅ {torch.__version__}")
        print(f"   CUDA available: {'✅' if torch.cuda.is_available() else '❌'}")
        if torch.cuda.is_available():
            print(f"   GPU device: {torch.cuda.get_device_name()}")
    except ImportError:
        print("   PyTorch: ❌ Not available")
    
    try:
        from transformers import AutoModel
        print("   Transformers: ✅")
    except ImportError:
        print("   Transformers: ❌ Not available")
    
    raise

## Phase 2: Prompt Testing

In [ ]:
# Test prompt loading (should use working 82% prompts)
print("📝 Testing prompt loading...")

try:
    # Test different document types
    for doc_type in ['invoice', 'receipt', 'bank_statement']:
        prompt = hybrid_processor.get_extraction_prompt(doc_type)
        print(f"\n📋 {doc_type.upper()} PROMPT:")
        print(f"  📏 Length: {len(prompt)} characters")
        print(f"  🔍 Preview: {prompt[:100]}...")
        
        # Check for working baseline characteristics (simple format)
        if "Extract structured data" in prompt:
            print(f"  ✅ Working baseline format detected")
        else:
            print(f"  ⚠️ May not be using working baseline prompts")
            
except Exception as e:
    print(f"❌ Prompt loading failed: {e}")
    traceback.print_exc()

## Phase 3: Single Image Test (Critical Test)

In [ ]:
# Find available test images
print("🔍 Looking for test images...")

# Common test image locations
possible_paths = [
    "evaluation_data/",
    "test_images/",
    "images/",
    "data/"
]

test_images = []
for base_path in possible_paths:
    if os.path.exists(base_path):
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.pdf']:
            import glob
            images = glob.glob(os.path.join(base_path, '**', ext), recursive=True)
            test_images.extend(images[:3])  # Limit to 3 per type

if test_images:
    print(f"📸 Found {len(test_images)} test images:")
    for i, img in enumerate(test_images[:5]):
        print(f"  {i+1}. {img}")
    
    # Use the first available image
    TEST_IMAGE_PATH = test_images[0]
    print(f"\n🎯 Using test image: {TEST_IMAGE_PATH}")
else:
    print("⚠️ No test images found. Please check image paths.")
    print("📂 Current directory contents:")
    import os
    for item in os.listdir('.')[:10]:
        print(f"  - {item}")

In [ ]:
# CRITICAL TEST: Single image processing
if 'TEST_IMAGE_PATH' in locals() and os.path.exists(TEST_IMAGE_PATH):
    print(f"🧪 CRITICAL TEST: Processing {TEST_IMAGE_PATH}")
    print("="*80)
    
    try:
        start_time = time.time()
        
        # Process single image with hybrid processor
        result = hybrid_processor.process_single_image(TEST_IMAGE_PATH)
        
        processing_time = time.time() - start_time
        
        print("\n🎉 HYBRID PROCESSOR SUCCESS!")
        print("="*80)
        print(f"⏱️ Processing time: {processing_time:.2f} seconds")
        print(f"📊 Fields extracted: {result['extracted_fields_count']}/{result['field_count']}")
        print(f"📈 Response completeness: {result['response_completeness']:.1%}")
        
        print("\n📋 EXTRACTED DATA:")
        for field, value in result['extracted_data'].items():
            status = "✅" if value != "NOT_FOUND" else "❌"
            print(f"  {status} {field}: {value}")
            
        print(f"\n📄 Raw response length: {len(result['raw_response'])} chars")
        
        # Check for ExtractionCleaner evidence
        if "🧹" in str(result) or any("🧹" in str(v) for v in result.values()):
            print("\n🧹 CLEANER ACTIVITY DETECTED - ExtractionCleaner is working!")
        
    except Exception as e:
        print(f"❌ CRITICAL TEST FAILED: {e}")
        import traceback
        traceback.print_exc()
        
else:
    print("⚠️ Skipping image test - no valid test image found")
    print("💡 To test with an image, upload one to the evaluation_data/ folder")

## Phase 4: Comparison Test (Optional)

In [ ]:
# Compare with current InternVL3 handler (if available)
print("🔄 COMPARISON TEST: Hybrid vs Current InternVL3 Handler")
print("="*80)

try:
    # Try to import current handler for comparison
    from internvl3_document_aware_handler import DocumentAwareInternVL3Handler
    
    # This would require setting up the current handler properly
    # For now, just show that the hybrid processor is working
    print("✅ Current handler available for comparison")
    print("💡 Manual comparison recommended:")
    print("   1. Run same image through current handler")
    print("   2. Compare accuracy and cleaner output")
    print("   3. Check for 🧹 CLEANER CALLED debug output")
    
except ImportError:
    print("ℹ️ Current handler not available for direct comparison")
    print("🎯 Hybrid processor results above show working implementation")

## Phase 5: Medium Field Test (If Basic Test Passes)

In [ ]:
# Test with more fields if basic test succeeded
print("🚀 EXTENDED TEST: Medium field list")
print("="*80)

if 'result' in locals() and result['extracted_fields_count'] > 0:
    print("✅ Basic test passed - proceeding with medium field test")
    
    try:
        # Create processor with medium field list
        medium_processor = DocumentAwareInternVL3HybridProcessor(
            field_list=TEST_FIELDS_MEDIUM,
            debug=True
        )
        
        if 'TEST_IMAGE_PATH' in locals() and os.path.exists(TEST_IMAGE_PATH):
            print(f"🧪 Processing with {len(TEST_FIELDS_MEDIUM)} fields...")
            
            medium_result = medium_processor.process_single_image(TEST_IMAGE_PATH)
            
            print(f"\n📊 Medium test results:")
            print(f"  Fields extracted: {medium_result['extracted_fields_count']}/{medium_result['field_count']}")
            print(f"  Response completeness: {medium_result['response_completeness']:.1%}")
            print(f"  Processing time: {medium_result['processing_time']:.2f}s")
            
            # Show additional fields found
            print("\n📋 Additional fields (vs small test):")
            for field in TEST_FIELDS_MEDIUM:
                if field not in TEST_FIELDS_SMALL:
                    value = medium_result['extracted_data'].get(field, 'NOT_FOUND')
                    status = "✅" if value != "NOT_FOUND" else "❌"
                    print(f"  {status} {field}: {value}")
        
    except Exception as e:
        print(f"❌ Medium test failed: {e}")
        print("💡 Basic processor still working - issue may be with increased field count")
        
else:
    print("⚠️ Skipping medium test - basic test did not extract any fields")
    print("💡 Focus on debugging basic functionality first")

## Summary and Next Steps

In [ ]:
# Test summary
print("🎯 HYBRID PROCESSOR TEST SUMMARY")
print("="*80)

# Check what tests passed
tests_passed = []
tests_failed = []

if 'hybrid_processor' in locals():
    tests_passed.append("✅ Processor initialization")
else:
    tests_failed.append("❌ Processor initialization")

if 'result' in locals() and result.get('extracted_fields_count', 0) > 0:
    tests_passed.append("✅ Single image processing")
    tests_passed.append("✅ Field extraction working")
else:
    tests_failed.append("❌ Single image processing")

if 'medium_result' in locals() and medium_result.get('extracted_fields_count', 0) > 0:
    tests_passed.append("✅ Medium field list processing")

print("\n🟢 PASSED TESTS:")
for test in tests_passed:
    print(f"  {test}")

if tests_failed:
    print("\n🔴 FAILED TESTS:")
    for test in tests_failed:
        print(f"  {test}")

print("\n🚀 NEXT STEPS:")
if len(tests_passed) >= 2:
    print("  1. ✅ Hybrid processor is working!")
    print("  2. 🎯 Test with full document field lists")
    print("  3. 📊 Compare accuracy with baseline (79%/77%/55%)")
    print("  4. 🔍 Look for 🧹 CLEANER CALLED output in verbose mode")
    print("  5. 📈 Target: Restore 82% accuracy baseline")
else:
    print("  1. 🔧 Debug initialization or image loading issues")
    print("  2. 📋 Check model path and GPU availability")
    print("  3. 🖼️ Verify test image format and accessibility")

print("\n💡 HYBRID ARCHITECTURE BENEFITS:")
print("  - InternVL3 model accuracy + Llama processing reliability")
print("  - ExtractionCleaner integration for value normalization")
print("  - Working 82% prompts (simple format)")
print("  - Zero risk to existing Llama pipeline")